# 02 — SIMON Reliability: Face Age vs Brain Age Across 73 Sessions

**Research question**: How does the variability of face age predictions (from MRI renders) 
compare to brain age variability across different scanners — for the *same individual*?

SIMON dataset: 1 subject (male, scanned at age 29–46), 73 sessions, 36 scanners, 13 MRI models.

This directly parallels the thesis work on brain age scanner variability, but adds the facial age axis.

**Outputs**:
- CV (coefficient of variation) for face age and brain age
- Bland-Altman-style scanner variability plots
- Scatter: face age vs session index (chronological order)

In [4]:
import sys
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
SIMON_DIR    = Path("../../data/simon")          # directory with all .mgz files
FACEAGE_ROOT = Path("../vendor/FaceAge")
RENDERS_DIR  = Path("../results/simon_renders")
OUT_DIR      = Path("../results/simon_reliability")
SKIN_LEVEL   = 40
BYPASS_MTCNN = True
DEVICE       = "cpu"   # change to 'cuda' on Colab with GPU
# ────────────────────────────────────────────────────────────────────────────

RENDERS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(Path(".").resolve().parent))

In [6]:
# ── 1. Build session table ───────────────────────────────────────────────────
from src.utils import load_simon_metadata

sessions = load_simon_metadata(SIMON_DIR)
print(f"Found {len(sessions)} sessions")
sessions.head(10)

Found 99 sessions


,session_id,age,run,manufacturer,scanner_model,acquisition_date,institution,filename,path
0,1,29.69,1,Philips,T5,2001-09-20,Montreal Neuro,sub-032633_ses-001_run-1_T1w.nii.gz,../../data/simon/images/sub-032633_ses-001_run...
1,2,30.43,1,Philips,T5,2002-06-17,Montreal Neuro,sub-032633_ses-002_run-1_T1w.nii.gz,../../data/simon/images/sub-032633_ses-002_run...
2,3,32.18,1,Philips,T5,2004-03-18,Montreal Neuro,sub-032633_ses-003_run-1_T1w.nii.gz,../../data/simon/images/sub-032633_ses-003_run...
3,4,34.52,1,Siemens,Symphony,2006-07-21,Hospital Sud Rennes,sub-032633_ses-004_run-1_T1w.nii.gz,../../data/simon/images/sub-032633_ses-004_run...
4,5,37.91,1,Philips,Achieva,2009-09-17,IRM Quebec - Mailloux,sub-032633_ses-005_run-1_T1w.nii.gz,../../data/simon/images/sub-032633_ses-005_run...
5,5,37.91,2,Philips,Achieva,2009-09-17,IRM Quebec - Mailloux,sub-032633_ses-005_run-2_T1w.nii.gz,../../data/simon/images/sub-032633_ses-005_run...
6,5,37.91,3,Philips,Achieva,2009-09-17,IRM Quebec - Mailloux,sub-032633_ses-005_run-3_T1w.nii.gz,../../data/simon/images/sub-032633_ses-005_run...
7,6,37.91,1,Philips,Achieva,2009-12-11,IRM Quebec - Mailloux,sub-032633_ses-006_run-1_T1w.nii.gz,../../data/simon/images/sub-032633_ses-006_run...
8,6,37.91,2,Philips,Achieva,2009-12-11,IRM Quebec - Mailloux,sub-032633_ses-006_run-2_T1w.nii.gz,../../data/simon/images/sub-032633_ses-006_run...
9,7,38.18,1,Philips,Achieva,2010-03-19,IRM Quebec - Mailloux,sub-032633_ses-007_run-1_T1w.nii.gz,../../data/simon/images/sub-032633_ses-007_run...


In [8]:
# ── 3. Run FaceAge on all renders ────────────────────────────────────────────
from src.face_age import predict_age_batch

good = sessions[sessions["render_ok"]].copy()
pngs = [Path(p) for p in good["render_png"]]

fa_results = predict_age_batch(
    img_paths=pngs,
    faceage_root=FACEAGE_ROOT,
    bypass_mtcnn=BYPASS_MTCNN,
    device=DEVICE,
)

import pandas as pd
fa_df = pd.DataFrame([{"render_png": r["path"], "face_age": r["age"], "mtcnn_found": r["mtcnn_found"]} for r in fa_results])
good = good.merge(fa_df, on="render_png", how="left")

print(f"Face age range: {good['face_age'].min():.1f} – {good['face_age'].max():.1f} years")
print(f"Face age mean ± std: {good['face_age'].mean():.1f} ± {good['face_age'].std():.1f}")

ModuleNotFoundError: No module named 'models'

In [ ]:
# ── 4. Run SynthSeg brain age (or load cached results) ───────────────────────
# If SynthSeg has already been run, point SYNTHSEG_CSV to the cached summary.
# Otherwise, run it now (requires FreeSurfer on PATH).

import numpy as np
SYNTHSEG_CSV = OUT_DIR / "brain_ages_synthseg.csv"

if SYNTHSEG_CSV.exists():
    brain_df = pd.read_csv(SYNTHSEG_CSV)
    print(f"Loaded cached SynthSeg results: {len(brain_df)} rows")
else:
    from src.brain_age import run_synthseg, synthseg_to_brain_age
    synthseg_dir = OUT_DIR / "synthseg_vols"
    brain_records = []
    for _, row in tqdm(good.iterrows(), total=len(good), desc="SynthSeg"):
        try:
            vol_csv = run_synthseg(row.path, synthseg_dir)
            ba = synthseg_to_brain_age(vol_csv)  # NaN until regressor trained
        except Exception as e:
            print(f"  ⚠ {row.filename}: {e}")
            ba = np.nan
        brain_records.append({"filename": row.filename, "brain_age": ba})
    brain_df = pd.DataFrame(brain_records)
    brain_df.to_csv(SYNTHSEG_CSV, index=False)

good = good.merge(brain_df[["filename", "brain_age"]], on="filename", how="left")

In [ ]:
# ── 5. Variability analysis ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

fa_cv = good["face_age"].std() / good["face_age"].mean() * 100
ba_cv = good["brain_age"].std() / good["brain_age"].mean() * 100

print(f"Face age  — CV = {fa_cv:.2f}%  (std={good['face_age'].std():.2f} yr)")
print(f"Brain age — CV = {ba_cv:.2f}%  (std={good['brain_age'].std():.2f} yr)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label, color in [
    (axes[0], "face_age",  "Face age (FaceAge on T1 render)",   "steelblue"),
    (axes[1], "brain_age", "Brain age (SynthSeg)",               "tomato"),
]:
    vals = good[col].dropna()
    ax.scatter(good.loc[vals.index, "session_id"], vals, alpha=0.6, color=color, s=20)
    ax.axhline(vals.mean(), color="black", linestyle="--", label=f"mean={vals.mean():.1f}")
    ax.axhspan(vals.mean() - vals.std(), vals.mean() + vals.std(), alpha=0.1, color=color, label=f"±1 SD ({vals.std():.1f} yr)")
    ax.set_xlabel("Session index")
    ax.set_ylabel("Predicted age (years)")
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle("SIMON (n=1 subject, 73 sessions) — scanner-induced variability", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "02_simon_variability.png", dpi=150)
plt.show()

In [ ]:
# ── 6. Save results table ────────────────────────────────────────────────────
out_csv = OUT_DIR / "simon_predictions.csv"
good[["session_id", "fs_version", "face_age", "brain_age", "mtcnn_found"]].to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")